# PrismalStateGraphBuilder

> Notebook generated from [`examples/extension/custom_subgraph.py`](../../examples/extension/custom_subgraph.py).

| Field | Value |
|------|-------|
| **Dataset** | Hard-coded AgentState |
| **API key** | ✅ **No API keys required** — uses stubs/mocks. |

**Expected result:** Compiles a mini-subgraph with 2 nodes and runs it. Prints the final messages.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Build and run a subgraph end-to-end with PrismalStateGraphBuilder.

Run::

    python examples/extension/custom_subgraph.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

from langchain_core.messages import AIMessage, HumanMessage

from prismal.agents.extension import PrismalStateGraphBuilder
from prismal.agents.state import create_initial_state


async def classify(state: dict) -> dict:
    text = state["messages"][-1].content
    return {"metadata": {"classify": {"len": len(text)}}}


async def respond(state: dict) -> dict:
    return {"messages": [AIMessage(content="Handled by the custom subgraph.")]}


async def main() -> None:
    builder = PrismalStateGraphBuilder("demo_pipeline")
    builder.add_node("classify", classify, capabilities=["general"])
    builder.add_node("respond", respond)
    builder.add_edge("classify", "respond")
    builder.add_edge("respond", "__end__")
    builder.set_entry_point("classify")

    # compile() -> SubgraphDefinition (registrable); compile_raw() -> runnable graph.
    compiled = builder.compile_raw()

    state = create_initial_state(session_id="demo")
    state["messages"] = [HumanMessage(content="Please summarise this.")]
    result = await compiled.ainvoke(state)
    print("final message:", result["messages"][-1].content)


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()